# Eyherabide, Rokem, Herz & Samengo (2008) on CRCNS ia-1

Paper: H. G. Eyherabide, A. Rokem, A. V. M. Herz and I. Samengo (2008), "Burst firing is a neural
code in an insect auditory system", Frontiers in Computational Neuroscience,
[doi:10.3389/neuro.10.003.2008](https://doi.org/10.3389/neuro.10.003.2008). The claims file
(`claims/crcns-ia-1/10.3389-neuro.10.003.2008.yaml`) was written from the version of record in
PubMed Central (PMC2525941), with its Methods and the legends of Figures 1 to 10 and Table 1; the
paper has no supplement and links no code.

Dataset: CRCNS data set ia-1, [doi:10.6080/K0BG2KWB](https://doi.org/10.6080/K0BG2KWB), read from
the public bucket `crcnsarchive` (`ia-1/crcns-ia1.zip`), streamed and opened in place.

Version: CRCNS data sets are not versioned, so the version is recorded as `unversioned` and the
file read is pinned by its MD5 sum in the data set's `checksums.md5`: `crcns-ia1.zip`
f0740514fabc6995fe54b2a361b6c763 (the helper checks the object's ETag against it before reading;
the object was last modified 2026-02-11). This is the only published state of the data set, so it
is the one pinned. Only `Data1` is used: the data descriptor names it as the data of this paper;
`Data2` (2004) is unpublished and used other stimuli.

| Claim | Tested here |
|---|---|
| data-sessions-and-trials | tested |
| table1-sessions-per-protocol | tested |
| firing-rate-uniform | tested |
| threshold-population | blocked: the thresholds are not in the data set |
| table1-bursting-sessions | tested |
| table1-isolated-spikes | tested |
| table1-burst-length | tested |
| fig5-burst-probability-vs-stimulus | tested |
| fig3-correlation-shape | tested (the data part; the model panels need the thresholds) |
| method-correlation-error | tested |
| model-matches-classification | blocked: the model needs the thresholds s_TH |
| fig6-bursts-less-correlated | tested |
| fig7ab-information-per-burst-increases | tested |
| fig7cd-occurrence-decreases | tested |
| fig7ef-jitter-independent-of-n | tested |
| fig7gh-information-rate | tested |
| direct-method-comparison | tested |
| fig8-nbta-differ | tested |
| fig8-latency-decreases | tested |
| fig8e-sigma-minimum | tested |
| sigma-n-below-sta-sigma | tested |
| fig9-nbta-not-concatenated-1bta | tested (data only; the model sessions need the thresholds) |
| fig9c-population-nbta | tested |
| fig10-burst-n-vs-height | tested |
| firing-rate-no-effect-on-limit | tested |
| model-reproduces-coding-trends | blocked: depends on model-matches-classification |

No step draws random numbers, so no seed is needed.

In [ ]:
import collections
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ia1

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("data set", ia1.CRCNS_ID, ia1.VERSION, "doi", ia1.DATASET_DOI, "zip md5", ia1.ZIP_MD5)
z = ia1.open_zip()
sessions = ia1.load_sessions(z)
cells = ia1.load_singlecells(z)
shapes = {name: ia1.load_stimulus_shape(z, name) for name in sorted({s.stimulus for s in sessions})}
cell_of = {e: i for i, row in enumerate(cells) for e in row}

## Data check

What was loaded, and the checks of the description against the values. The data descriptor
says: spike times in microseconds from the stimulus trigger; trials of the first 1000 ms of the
stimulus, 98 to 533 per stimulus; 700 ms pauses; two interleaved stimuli per experiment;
`cell.dat` gives the stimulus names and intensities (the maximal intensity, mean plus 3 SD);
stimulus files at 20 kHz, 10 s long; `singlecells.txt` lists the experiments made on one cell.

In [ ]:
rows = []
for s in sessions:
    r = ia1.trial_rates(s.trials)
    rows.append(dict(session=s.name, exp=s.exp, date=s.exp[:8], cell=cell_of.get(s.exp, -1),
                     stimulus=s.stimulus, protocol=s.protocol, cutoff=s.cutoff, intensity=s.intensity,
                     header_intensity=float(s.header["intensity (dB)"]) if "intensity (dB)" in s.header else np.nan,
                     desired_rate=float(s.header["desired firing rate (Hz)"]) if "desired firing rate (Hz)" in s.header else np.nan,
                     pause_ms=float(s.header["pause between stimuli (msec)"]) if "pause between stimuli (msec)" in s.header else np.nan,
                     header_stim=s.header.get("stimulus file", "").split("/")[-1].removesuffix(".dat"),
                     offline=s.offline, n_trials=s.n_trials, rate=r.mean(), rate_sd=r.std(ddof=1),
                     rate_full=ia1.trial_rates(s.trials, 0, 1).mean(), rate_sd_full=ia1.trial_rates(s.trials, 0, 1).std(ddof=1)))
D = pd.DataFrame(rows)
print("Data1 experiments:", D.exp.nunique(), " sessions:", len(D), " offline-sorted sessions (no OEL header):", D.offline.sum())
print("experiments not listed in singlecells.txt:", sorted(set(D.exp) - set(cell_of)))
print("cells (rows of singlecells.txt, plus experiments not listed):", len(cells) + len(set(D.exp) - set(cell_of)),
      " recording dates:", D.date.nunique())
print("\nSessions by stimulus file (cell.dat):")
print(D.groupby("stimulus").agg(sessions=("session", "size"), protocol=("protocol", "first")).to_string())
hdr = D[D.header_stim != ""]
print("\nHeader stimulus agrees with cell.dat in", (hdr.header_stim.str.replace("gauss_", "") == hdr.stimulus.str.replace("gauss_", "")).sum(),
      "of", len(hdr), "sessions with a stimulus line in the header")
d = (D.header_intensity - D.intensity).dropna()
print("header intensity minus cell.dat intensity (dB): n =", len(d), " median |diff|", round(d.abs().median(), 2),
      " max |diff|", round(d.abs().max(), 2), " sessions over 1 dB:", list(D.session[(D.header_intensity - D.intensity).abs() > 1]))
print("desired firing rate in headers:", D.desired_rate.value_counts(dropna=False).to_dict())
print("pause between stimuli (ms) in headers:", D.pause_ms.value_counts(dropna=False).to_dict())
allsp = np.concatenate([np.concatenate(s.trials) for s in sessions])
print("\nspike times (s): min", allsp.min(), " max", allsp.max(), " fraction after 1 s (in the pause):", round((allsp > 1).mean(), 4))
print("trials per session: min", D.n_trials.min(), " max", D.n_trials.max())
print("rate over 200-1000 ms (Hz): min", round(D.rate.min(), 1), " max", round(D.rate.max(), 1))
unsorted = sum(int((np.diff(t) <= 0).any()) for s in sessions for t in s.trials)
print("trials with unsorted or repeated spike times:", unsorted)
isi = np.concatenate([np.diff(t) for s in sessions for t in s.trials])
print("shortest ISI (ms):", round(isi.min() * 1e3, 2), " 0.1 ms quantile of ISIs (ms):", round(np.quantile(isi, 0.001) * 1e3, 2))
for name, sh in shapes.items():
    print(f"stimulus {name}: {len(sh)} samples in the first 1.05 s at 20 kHz, dB re its 10 s maximum, range {sh.min():.1f} to {sh.max():.1f}, SD over 0-1 s {sh[:20000].std():.2f} dB")
# Duplicate recordings: identical spike trains or identical first trials.
seen = collections.defaultdict(list)
for s in sessions:
    seen[np.concatenate(s.trials).round(6).tobytes()].append(s.name)
    seen[("first", s.trials[0].round(6).tobytes())].append(s.name)
print("sessions with identical spike times or identical first trials:", [v for v in seen.values() if len(v) > 1])

The stimulus files are named for their SD and cutoff (`gauss_st6_co200` is SD 6 dB, cutoff 200
Hz). Their SDs over the first second can be checked against the names (6 dB files near 6 dB).

**Clock check.** Spikes and stimuli share the trial's clock if the spike-triggered average of the
stimulus (STA) shows the upstroke a few ms before the spike, which the paper describes (Figure 1:
"responses typically appear after stimulus upstrokes with an delay of 3.4 msec"). The contrast is
the same STA computed from the stimulus shifted by 100 ms, which keeps every property of the
stimulus and the spike train except their alignment.

In [ ]:
P = [s for s in sessions if s.protocol]          # sessions in the paper's seven protocols


def stim_db(s):
    return shapes[s.stimulus] + s.intensity


def spikes_in(s, t0=ia1.T0, t1=ia1.T1):
    return np.concatenate([t[(t >= t0) & (t < t1)] for t in s.trials])


rows = []
for s in P:
    st = stim_db(s)
    sp = spikes_in(s)
    seg, tt = ia1.segments(st, sp)
    sta = seg.mean(0) - st[4000:20000].mean()
    seg2, _ = ia1.segments(st, sp - 0.1)
    sta2 = seg2.mean(0) - st[4000:20000].mean()
    rows.append(dict(session=s.name, protocol=s.protocol, sta_peak=sta.max(), lat_ms=-tt[np.argmax(sta[tt <= 0])] * 1e3,
                     shifted_peak=sta2.max()))
C = pd.DataFrame(rows)
print("STA peak above the mean stimulus (dB): median", round(C.sta_peak.median(), 2), " range", round(C.sta_peak.min(), 2), "to", round(C.sta_peak.max(), 2))
print("with the stimulus shifted by 100 ms: median", round(C.shifted_peak.median(), 2), " range", round(C.shifted_peak.min(), 2), "to", round(C.shifted_peak.max(), 2))
print("sessions whose STA peak exceeds the shifted one:", (C.sta_peak > C.shifted_peak).sum(), "of", len(C))
print("latency of the STA maximum (ms): median", C.lat_ms.median(), " range", C.lat_ms.min(), "to", C.lat_ms.max())
print("sessions where the true STA peak is under 1 dB:", list(C.session[C.sta_peak < 1]))

## Session selection and burst identification

Kept sessions (claim data-sessions-and-trials). The paper keeps sessions whose trial-to-trial SD
of the firing rate is below 35 Hz and whose mean rate is about 100 Hz. Readings, from the
criterion: (a) the SD condition alone; (b) the SD condition and a target rate of 100 Hz where
the header records one. Per-trial rates over 200 to 1000 ms, and over the whole 1 s as a
variant. Only sessions of the paper's seven protocols (Table 1) can be kept; the data set's other
stimuli (`gaussModality_co200`, `lmd_co200`, cutoffs 50 and 1600 Hz) are not in the paper's design
(Methods: "The other signal could be one of six different stimulation protocols").

Bursts, by the Methods (Burst identification), under every combination of the readings of the
open details (claims file, opening comment): peak (a) first local maximum, (b) largest value in
0 < tau < 5 ms; minimum (a) first significant local minimum after the peak, (b) lowest value up
to 1.25 / cutoff; error bar N (a) the number of product terms, (b) the number of trials, and (c)
sigma the SD over trials of each trial's C_s with N the number of trials (added here, see
`ia1.py`).

In [ ]:
def kept(reading="a", window="200-1000"):
    col = "rate_sd" if window == "200-1000" else "rate_sd_full"
    out = []
    for s in P:
        row = D.loc[D.session == s.name].iloc[0]
        ok = row[col] < 35
        if reading == "b" and not np.isnan(row.desired_rate):
            ok = ok and row.desired_rate == 100
        if ok:
            out.append(s)
    return out


KEPT = kept("a")
COMBOS = list(itertools.product("ab", "ab", "abc"))
MAIN = ("b", "a", "a")
limits = {c: {s.name: ia1.burst_limit(s.trials, s.cutoff, *c)[0] for s in P} for c in COMBOS}
for c in COMBOS:
    lim = limits[c]
    nb = sum(lim[s.name] > 0 for s in KEPT)
    print("peak %s, minimum %s, N %s: bursting %3d of %d kept; median limiting ISI %s ms" % (
        *c, nb, len(KEPT), np.round(np.median([lim[s.name] for s in KEPT if lim[s.name] > 0]) * 1e3, 2) if nb else "-"))
LIM = limits[MAIN]
BURSTING = [s for s in KEPT if LIM[s.name] > 0]
print("\nmain reading", MAIN, ": kept", len(KEPT), " bursting", len(BURSTING))

## data-sessions-and-trials

> Thirty-seven receptor cells were recorded from 23 animals, giving 132 sessions, of which 86 met
> the firing-rate and stationarity conditions; each stimulus was repeated 98 to 503 times, 172 on
> average.

Data1 does not record the animal. The recording date is shown as a stand-in (one animal per
day is an assumption the descriptor does not make); cells are the rows of `singlecells.txt`.

In [ ]:
def describe(sel, label):
    names = {s.name for s in sel}
    d = D[D.session.isin(names)]
    # cells: rows of singlecells.txt, plus each experiment it does not list
    return dict(population=label, cells=d.cell[d.cell >= 0].nunique() + d.exp[d.cell < 0].nunique(),
                dates=d.date.nunique(), sessions=len(d), min_trials=d.n_trials.min(), max_trials=d.n_trials.max(),
                mean_trials=round(d.n_trials.mean(), 1), mean_rate_sd=round(d.rate_sd.mean(), 2),
                mean_rate_sd_full=round(d.rate_sd_full.mean(), 2))


tab = pd.DataFrame([describe(sessions, "all Data1 sessions"),
                    describe([s for s in sessions if s.stimulus.replace("gauss_", "").startswith("st")], "Gaussian AM stimuli (any cutoff)"),
                    describe(P, "seven protocols"),
                    describe(kept("a"), "kept (a), 200-1000 ms"), describe(kept("b"), "kept (b), 200-1000 ms"),
                    describe(kept("a", "full"), "kept (a), 0-1000 ms"), describe(kept("b", "full"), "kept (b), 0-1000 ms")])
print(tab.to_string(index=False))
print("\nsessions in the seven protocols with rate SD >= 35 Hz (200-1000 ms):",
      list(D.session[D.protocol.notna() & (D.rate_sd >= 35)]))
print("SD of the trial rates over the seven-protocol sessions (Hz): quantiles 0, 0.25, 0.5, 0.75, 0.9, 1:",
      np.quantile(D.rate_sd[D.protocol.notna()], [0, .25, .5, .75, .9, 1]).round(1))
expected = dict(cells=(37, 3.7), sessions_recorded=(132, 13.2), kept=(86, 8.6), min_trials=(98, 9.8), max_trials=(503, 50.3),
                mean_trials=(172, 17.2), mean_rate_sd=(6, 0.6))
print("\nexpected (value, tolerance):", expected)

**Result: partially reproduced.** The 116 sessions of the seven protocols come from 37 cells
recorded on 23 dates, the paper's 37 cells and 23 animals (if one animal was recorded per date, an
assumption, as Data1 does not record the animal). The 122 Gaussian AM sessions (any cutoff) are
within 10% of the paper's 132 sessions; all 150 Data1 sessions are not. The selection does not
reproduce: the SD < 35 Hz condition keeps 115 of the 116 sessions (113 with the 100 Hz target of
reading (b)), against 86, so the paper's selection removed sessions by a rule it does not state.
Over the kept sessions the trials range from 69 (outside 98 +/- 9.8) to 503 and average 160.3
(within 172 +/- 17.2), and the mean trial-to-trial SD of the rate is 6.48 Hz (within 6 +/- 0.6).

## table1-sessions-per-protocol

> The 86 kept sessions divide among the seven stimulus protocols as 43 (6 dB, 200 Hz), 6 (3 dB,
> 200 Hz), 7 (12 dB, 200 Hz), 7 (6 dB, 25 Hz), 8 (6 dB, 100 Hz), 7 (6 dB, 400 Hz) and 8 (6 dB, 800 Hz).

In [ ]:
T1_SESS = [43, 6, 7, 7, 8, 7, 8]
rows = []
for label, sel in [("kept (a) 200-1000", kept("a")), ("kept (b) 200-1000", kept("b")),
                   ("kept (a) 0-1000", kept("a", "full")), ("kept (b) 0-1000", kept("b", "full")), ("seven protocols", P)]:
    cnt = collections.Counter(s.protocol for s in sel)
    vals = [cnt[p] for p in range(1, 8)]
    rows.append(dict(population=label, **{f"p{p}": v for p, v in zip(range(1, 8), vals)},
                     within_10pct=sum(abs(v - e) <= 0.1 * e for v, e in zip(vals, T1_SESS))))
rows.append(dict(population="paper", **{f"p{p}": v for p, v in zip(range(1, 8), T1_SESS)}, within_10pct=7))
print(pd.DataFrame(rows).to_string(index=False))
exps = collections.defaultdict(list)
for s in P:
    exps[s.exp].append(s.protocol)
print("\nexperiments with both stimuli in the protocols, by pair:",
      dict(collections.Counter(tuple(sorted(v)) for v in exps.values() if len(v) == 2)))

**Result: not reproduced.** None of the seven counts is within 10% of Table 1 under any reading:
the kept sessions are 60, 9, 8, 10, 11, 8 and 9 per protocol (reading (a), 200 to 1000 ms) against
43, 6, 7, 7, 8, 7 and 8. The data set holds every reference session paired with each other
protocol (55 experiments) and more reference sessions than the paper counts; the paper's 86 are a
subset whose selection is not stated.

## firing-rate-uniform

> The kept sessions had a mean firing rate of 113 Hz (SD 16 Hz) that did not differ significantly
> across stimulus conditions (one-way ANOVA, p = 0.58).

In [ ]:
rows = []
for label, sel, col in [("kept (a), 200-1000 ms", kept("a"), "rate"), ("kept (a), 0-1000 ms", kept("a", "full"), "rate_full"),
                        ("all seven-protocol sessions, 200-1000 ms", P, "rate"), ("all seven-protocol sessions, 0-1000 ms", P, "rate_full"),
                        ("kept (b), 200-1000 ms", kept("b"), "rate")]:
    d = D[D.session.isin({s.name for s in sel})]
    groups = [g[col].values for _, g in d.groupby("protocol")]
    F, p = stats.f_oneway(*groups)
    rows.append(dict(population=label, n=len(d), mean_rate=round(d[col].mean(), 1), sd_rate=round(d[col].std(ddof=1), 1),
                     F=round(F, 2), p=round(p, 3)))
print(pd.DataFrame(rows).to_string(index=False))
d = D[D.session.isin({s.name for s in KEPT})]
print("\nmean rate by protocol (kept (a), 200-1000 ms):", d.groupby("protocol").rate.mean().round(1).to_dict())

**Result: reproduced.** Over the 115 kept sessions the mean rate is 110.2 Hz (within 113 +/- 16)
with SD 16.7 Hz (within 16 +/- 1.6), and a one-way ANOVA across the seven protocols finds no
difference (F = 0.49, p = 0.818). Over all seven-protocol sessions and with whole-trial rates the
ANOVA gives p = 0.809 and 0.471; whole-trial rates average 114.7 Hz (SD 16.9). The population is
115 sessions, not the paper's 86.

## threshold-population (blocked)

> The mean threshold of the cells at their best frequency was 58 dB (SD 14 dB).

Data1 holds spike times, `cell.dat` (stimulus names and the intensities they were played at) and
the stimuli. It holds no threshold, best-frequency or tuning (f-I) measurement: the listing below
shows every kind of file in Data1. Blocked, `data_missing`.

In [ ]:
kinds = collections.Counter()
for n in z.namelist():
    if n.startswith(ia1.ROOT) and not n.endswith("/"):
        base = n.split("/")[-1]
        kinds[base.split("_")[-1] if "_" in base and n.count("/") > 3 and "stimuli" not in n else ("stimuli/" + base if "stimuli" in n else base)] += 1
print(dict(kinds))
print("cell.dat keys:", sorted({line.split(":")[0].strip() for n in z.namelist() if n.endswith("cell.dat")
                               for line in z.read(n).decode("latin1").splitlines() if ":" in line}))

## table1-bursting-sessions

> The number of sessions with bursts of n > 1 depends on the stimulus, from none for 3 dB and for
> 800 Hz to all 7 for 12 dB and 7 of 8 for 100 Hz; 40 of 43 reference sessions, 6 of 7 at 25 Hz
> and 3 of 7 at 400 Hz.

Counted over the kept sessions, under every combination of the burst-identification readings. The
fraction of sessions is shown beside the counts, since the counts depend on how many sessions are
kept.

In [ ]:
T1_BURST = [40, 0, 7, 6, 7, 3, 0]
tol = [max(0.1 * e, 0) for e in T1_BURST]


def n_gt1_sessions(sel, lim):
    cnt = collections.Counter()
    for s in sel:
        if lim[s.name] > 0 and (ia1.burst_counts(s.trials, lim[s.name])[1:] > 0).any():
            cnt[s.protocol] += 1
    return [cnt[p] for p in range(1, 8)]


rows = []
for kr in ["a", "b"]:
    sel = kept(kr)
    tot = collections.Counter(s.protocol for s in sel)
    for c in COMBOS:
        v = n_gt1_sessions(sel, limits[c])
        rows.append(dict(kept=kr, readings="".join(c), **{f"p{p}": f"{x}/{tot[p]}" for p, x in zip(range(1, 8), v)},
                         within=sum(abs(x - e) <= t for x, e, t in zip(v, T1_BURST, tol))))
print(pd.DataFrame(rows).to_string(index=False))
print("paper:", [f"{a}/{b}" for a, b in zip(T1_BURST, T1_SESS)])

**Result: not reproduced.** Under the main reading (peak b, minimum a, N a) the kept sessions
with bursts of n > 1 are 52/60, 2/9, 8/8, 9/10, 10/11, 3/8 and 0/9 against 40/43, 0/6, 7/7, 6/7,
7/8, 3/7 and 0/8: 2 of 7 counts within tolerance (400 and 800 Hz), and no combination of readings
does better. The pattern across protocols is the paper's (most reference, 12 dB, 25 Hz and 100 Hz
sessions burst; few at 400 Hz; none at 800 Hz), but the counts are higher because more sessions
are kept, and 2 of 9 sessions at 3 dB burst where the paper has none. Under peak reading (a) only
6/60 reference sessions burst, and with N the number of trials (b) none does.

## table1-isolated-spikes

> The percentage of isolated spikes among all bursts of the bursting sessions depends on the
> stimulus (74, 100, 62, 60, 55, 96 and 100% for protocols 1 to 7), and in every protocol isolated
> spikes are the most probable burst.

In [ ]:
T1_ISO = [74, 100, 62, 60, 55, 96, 100]


def protocol_counts(sel, lim):
    # Pooled n counts per protocol over the bursting sessions (all sessions where a protocol has none).
    out = {}
    for p in range(1, 8):
        ses = [s for s in sel if s.protocol == p]
        bur = [s for s in ses if lim[s.name] > 0]
        use = bur if bur else ses
        tot = np.zeros(400)
        per = []
        for s in use:
            c = ia1.burst_counts(s.trials, lim[s.name])
            tot[:len(c)] += c
            per.append(c[0] / c.sum())
        out[p] = (np.trim_zeros(tot, "b"), per, len(bur))
    return out


rows = []
PC = {}
for c in COMBOS:
    pc = protocol_counts(KEPT, limits[c])
    PC[c] = pc
    iso = [100 * pc[p][0][0] / pc[p][0].sum() for p in range(1, 8)]
    iso_avg = [100 * np.mean(pc[p][1]) for p in range(1, 8)]
    most = [int(np.argmax(pc[p][0]) + 1) for p in range(1, 8)]
    rows.append(dict(readings="".join(c), **{f"p{p}": round(v, 1) for p, v in zip(range(1, 8), iso)},
                     within=sum(abs(v - e) <= 0.1 * e for v, e in zip(iso, T1_ISO)), most_probable_n=most,
                     session_average=[round(v, 1) for v in iso_avg]))
print(pd.DataFrame(rows).to_string(index=False))
print("paper:", T1_ISO)

**Result: partially reproduced.** Under the main reading the pooled percentage of isolated
spikes is 79.5, 77.7, 67.7, 78.1, 73.5, 95.0 and 100% against 74, 100, 62, 60, 55, 96 and 100%: 4
of 7 within tolerance (protocols 1, 3, 6, 7); 3 dB, 25 Hz and 100 Hz have fewer isolated spikes
than the paper at 3 dB and more at 25 and 100 Hz. No combination of readings brings more than 4
within tolerance. n = 1 is the most probable burst in every protocol under the main reading, as
the paper says.

## table1-burst-length

> The highest n and the average n depend on the stimulus; the longest bursts (up to 15 spikes,
> average 2.10 with SD 3.67) occur at 25 Hz, and no bursts occur at 3 dB or at 800 Hz.

In [ ]:
EXP_HI = [9, 1, 8, 15, 9, 3, 1]
EXP_AV = [1.3, 1, 1.5, 2.1, 1.7, 1.04, 1]
EXP_SD = {1: 1.3, 3: 1.6, 4: 3.67, 5: 1.8, 6: 0.42}
rows = []
for c in COMBOS:
    pc = PC[c]
    hi = [len(pc[p][0]) for p in range(1, 8)]
    av = [ia1.average_n(pc[p][0])[0] for p in range(1, 8)]
    sd = [ia1.average_n(pc[p][0])[1] for p in range(1, 8)]
    ok = (sum(abs(h - e) <= 0.1 * e for h, e in zip(hi, EXP_HI)) + sum(abs(a - e) <= 0.1 * e for a, e in zip(av, EXP_AV))
          + sum(abs(sd[p - 1] - e) <= 0.1 * e for p, e in EXP_SD.items()))
    rows.append(dict(readings="".join(c), highest_n=hi, average_n=[round(a, 2) for a in av], sd_n=[round(x, 2) for x in sd], within_of_19=ok))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: highest", EXP_HI, " average", EXP_AV, " SD", EXP_SD)

**Result: not reproduced.** Under the main reading 7 of the 19 values are within tolerance
(highest n 7, 9, 8, 14, 7, 5, 1 against 9, 1, 8, 15, 9, 3, 1; average n 1.25, 1.30, 1.46, 1.85,
1.36, 1.05, 1.00 against 1.3, 1, 1.5, 2.1, 1.7, 1.04, 1), below the 10 the criterion needs for a
partial reproduction, and no combination of readings gives more than 7. The longest bursts are at
25 Hz (14 spikes) and 800 Hz has none, as in the paper, but 3 dB has bursts of up to 9 spikes in
its 2 bursting sessions.

## fig5-burst-probability-vs-stimulus

> The probability of generating isolated spikes is lowest, and high-n bursts most frequent, for
> large amplitude fluctuations (12 dB) and cutoff frequencies around 100 Hz.

In [ ]:
def pn_table(sel, lim, bursting_only):
    rows = []
    for s in sel:
        if bursting_only and lim[s.name] <= 0:
            continue
        c = ia1.burst_counts(s.trials, lim[s.name]).astype(float)
        c = np.r_[c, np.zeros(3)][:3] / c.sum()
        rows.append(dict(protocol=s.protocol, P1=c[0], P2=c[1], P3=c[2]))
    return pd.DataFrame(rows).groupby("protocol").agg(["mean", "std", "size"])


for c in [MAIN, ("b", "b", "a"), ("b", "a", "c"), ("a", "a", "a")]:
    for reading, bo in [("(a) all kept", False), ("(b) bursting only", True)]:
        t = pn_table(KEPT, limits[c], bo)
        means = t.xs("mean", axis=1, level=1)
        cut = {25: 4, 100: 5, 200: 1, 400: 6, 800: 7}
        sdp = {3: 2, 6: 1, 12: 3}
        p1c = {k: means.P1.get(v, np.nan) for k, v in cut.items()}
        p1s = {k: means.P1.get(v, np.nan) for k, v in sdp.items()}
        print(f"readings {''.join(c)}, sessions {reading}")
        print("  P(n=1) by cutoff:", {k: round(v, 3) for k, v in p1c.items()}, " lowest at", min(p1c, key=p1c.get), "Hz")
        print("  P(n=1) by SD:", {k: round(v, 3) for k, v in p1s.items()}, " lowest at", min(p1s, key=p1s.get), "dB")
        for col in ["P2", "P3"]:
            print(f"  {col} by cutoff:", {k: round(means[col].get(v, np.nan), 3) for k, v in cut.items()},
                  " by SD:", {k: round(means[col].get(v, np.nan), 3) for k, v in sdp.items()})
t = pn_table(KEPT, LIM, False)
print("\nmain reading, all kept: mean, SD and number of sessions per protocol")
print(t.round(3).to_string())

**Result: reproduced.** Under the main reading the population average of P(n = 1) is lowest at
100 Hz among the cutoffs (0.755, against 0.806 at 25, 0.820 at 200, 0.982 at 400 and 1.000 at 800
Hz) and lowest at 12 dB among the SDs (0.686, against 0.820 at 6 and 0.957 at 3 dB), with all kept
sessions and with bursting sessions only. P(n = 2) and P(n = 3) are highest at 100 Hz and at 12 dB
over all kept sessions. With the bounding minimum read as the lowest value (b) the lowest P(n = 1)
moves to 25 Hz, where that reading gives very long bursts.

## fig3-correlation-shape

> For a cell tested with all stimuli, larger stimulus SD gives correlation functions with sharper
> peaks, and faster stimuli lead from multi-modal (25 Hz) to single-peaked (100, 200 Hz) to
> shallower and broader correlation functions (400, 800 Hz).

`singlecells.txt` lists one cell recorded in seven experiments, 03-04-23-ad to -aj, which together
hold all seven protocols (six reference sessions, one of each other protocol): this is taken as the
cell of Figure 3. C_s is normalised by the squared mean rate so that sessions compare. "Sharper"
is read as (a) a higher first peak over the baseline (the mean of C_s/rbar^2 over 20 to 40 ms) and
(b) a narrower first peak (full width at half height above that baseline). Peaks are counted on
C_s/rbar^2 smoothed with a 1 ms running mean (a standard choice that keeps the ms-scale shape), as
local maxima within 0 < tau <= 1.25 / cutoff that rise above the neighbouring minima by more than a
prominence threshold; the threshold is not in the paper, so 0.05, 0.1 and 0.2 are shown.

In [ ]:
from scipy.signal import find_peaks

fig3_cell = next(row for row in cells if "03-04-23-ad" in row)
F3 = [s for s in P if s.exp in fig3_cell]
rows = []
curves = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for s in F3:
    tau, c, sd, count, ntr, sdt = ia1.correlation(s.trials, 0.06)
    rb = ia1.binned(s.trials).astype(bool).mean() / ia1.DT
    cn = c / rb ** 2
    base = cn[(tau >= 0.02) & (tau <= 0.04)].mean()
    sm = np.convolve(cn, np.ones(10) / 10, mode="same")
    sm[:6] = cn[:6]
    lim_i = int(1.25 / s.cutoff / ia1.DT)
    ip = 1 + np.argmax(cn[1:51])
    height = cn[ip] - base
    width = ia1.fwhm(cn, ip, base) * ia1.DT * 1e3
    npk = {}
    for prom in [0.05, 0.1, 0.2]:
        pk, _ = find_peaks(sm[1:lim_i + 1], prominence=prom)
        npk[prom] = len(pk)
    rows.append(dict(session=s.name, sd=s.sd_cut[0], cutoff=s.cutoff, bursting=LIM[s.name] > 0,
                     limit_ms=round(LIM[s.name] * 1e3, 1), peak_ms=round(tau[ip] * 1e3, 1), peak_height=round(height, 2),
                     peak_fwhm_ms=round(width, 1), **{f"peaks_prom{k}": v for k, v in npk.items()}))
    curves.append(cn[1:])
    ax = axes[0] if s.sd_cut[1] == 200 else axes[1]
    ax.plot(tau[1:] * 1e3, cn[1:], lw=0.8, label=f"{s.sd_cut[0]} dB, {s.cutoff} Hz ({s.exp[-2:]})")
F3T = pd.DataFrame(rows)
print(F3T.to_string(index=False))
for ax, t in zip(axes, ["cutoff 200 Hz", "SD 6 dB"]):
    ax.set(xlabel="tau (ms)", ylabel="C_s / rbar^2", title=t, ylim=(-1.5, 4))
    ax.legend(fontsize=7)
allc = np.concatenate(curves)
print("C_s/rbar^2 for tau > 0: range", allc.min().round(2), "to", allc.max().round(2), "; points outside the axes (-1.5 to 4):", int(((allc < -1.5) | (allc > 4)).sum()))
plt.show()
ref = F3T[(F3T.sd == 6) & (F3T.cutoff == 200)]
print("reference sessions: peak height range", ref.peak_height.min(), "to", ref.peak_height.max(), "; FWHM range", ref.peak_fwhm_ms.min(), "to", ref.peak_fwhm_ms.max(), "ms")

**Result: reproduced** (resting on the peak-counting threshold). In the cell recorded with all
seven protocols (03-04-23-ad to -aj) the first peak of C_s/rbar^2 rises above its baseline by 0.50
at 3 dB, 1.14 to 2.35 in the six 6 dB sessions and 5.06 at 12 dB, so it sharpens with SD under
reading (a); its width (b) does not order (0.4, 0.4 to 0.5, 0.3 ms). At 25 Hz C_s has 4 to 14
peaks, at 200 Hz one; at 100 Hz one peak at prominence 0.2 and two at 0.05 and 0.1. The 3 dB, 400
Hz and 800 Hz sessions have no separable first peak, as the paper names for Figures 3A, E and F.

## method-correlation-error

> The temporal average of the estimated error of the correlation function was on average 3.4% (SD
> 1.5%) of the span of C_s.

Delta = sigma / sqrt(N) at each tau, averaged over tau, over max minus min of C_s, averaged over
kept sessions. Readings: sigma and N (a), (b) and (c); tau range (a) 0 to 1.25 / cutoff, (b) the
whole range up to 800 ms. Whether tau = 0, where C_s holds the self term (rbar / dt, two orders of
magnitude above the rest), enters the span is not stated: both are shown.

In [ ]:
rows = []
for s in KEPT:
    tau, c, sd, count, ntr, sdt = ia1.correlation(s.trials, 0.7999)
    L = int(round(1.25 / s.cutoff / ia1.DT))
    for nm in "abc":
        err = ia1.correlation_error(sd, count, ntr, sdt, nm)
        for rng_name, sl in [("(a) 0 to 1.25/cutoff", slice(0, L + 1)), ("(b) 0 to 800 ms", slice(0, None))]:
            for z0 in [True, False]:
                ss = sl if z0 else slice(1, sl.stop)
                rows.append(dict(session=s.name, N=nm, tau_range=rng_name, tau0=z0,
                                 pct=100 * err[ss].mean() / (c[ss].max() - c[ss].min())))
E = pd.DataFrame(rows)
print(E.groupby(["N", "tau_range", "tau0"]).pct.agg(["mean", "std", "size"]).round(2).to_string())

**Result: reproduced** (resting on leaving tau = 0 out of the span). With sigma the SD of the
product terms and N their number (reading a), over the whole range of lags and without the
zero-lag self term, the error averages 2.69% (SD 1.37%) of the span of C_s, within 3.4 +/- 1.5%;
reading (c) gives 2.67%. With tau = 0 in the span every reading is below 0.2% or, with N the
number of trials, 6.5 to 7.9%; over 0 to 1.25 / cutoff without tau = 0 the values are unstable
(means 30 to 4000%).

## model-matches-classification (blocked)

> A threshold-linear Poisson model with refractoriness and no free parameters classifies sessions
> as bursting or not in agreement with the real data in 86% of sessions, and where both are
> bursting their limiting ISIs differ by less than 1 ms in 81% of cases.

The model thresholds the stimulus at the cell's measured threshold s_TH before filtering (Eq. 2),
and the criterion says the claim is not testable as the paper ran it if the data set does not hold
s_TH. Data1 holds no threshold (see threshold-population). Blocked, `data_missing`; the dependent
model-reproduces-coding-trends is blocked for the same reason.

## fig6-bursts-less-correlated

> Representing the spike train as n-bursts reduces correlations between time bins; the mean
> squared Pearson coefficient over t from 200 to 990 ms and tau from 0 to 10 ms is 2.89 times
> larger (SD 1.49) for spikes than for bursts, on average over the bursting sessions.

Eq. 11 across trials at each (t, tau). Bin width (a) 0.1 ms, (b) 0.4 ms; tau = 0 included or not.
The spike variable is 1 when the bin holds a spike. Bursting sessions under the main burst
reading; detail (4) reading (b) (bursting sessions that have bursts of n > 1) is shown too.

In [ ]:
rows = []
for s in BURSTING:
    for bw in [1e-4, 4e-4]:
        xs = (ia1.binned(s.trials, dt=bw) > 0).astype(np.int8)
        xb = ia1.burst_matrix(s.trials, LIM[s.name], dt=bw)
        nl, te = int(round(0.01 / bw)), int(round(0.79 / bw))
        cs, cb = ia1.sq_pearson_by_lag(xs, nl, te), ia1.sq_pearson_by_lag(xb, nl, te)
        has_n2 = (xb > 1).any()
        rows.append(dict(session=s.name, bin_ms=bw * 1e3, n_gt1=has_n2, with_tau0=cs.mean() / cb.mean(),
                         without_tau0=cs[1:].mean() / cb[1:].mean()))
R6 = pd.DataFrame(rows)
for (bw, n2), g in R6.groupby(["bin_ms", "n_gt1"]):
    pass
summ = []
for bw in [0.1, 0.4]:
    for lab, g in [("(a) all bursting", R6[R6.bin_ms == bw]), ("(b) with n > 1", R6[(R6.bin_ms == bw) & R6.n_gt1])]:
        for col in ["with_tau0", "without_tau0"]:
            summ.append(dict(bin_ms=bw, sessions=lab, tau0=col, n=len(g), mean_ratio=round(g[col].mean(), 2),
                             sd_ratio=round(g[col].std(ddof=1), 2), frac_above_1=round((g[col] > 1).mean(), 3),
                             min=round(g[col].min(), 2), max=round(g[col].max(), 2)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: reproduced.** Over the 84 bursting sessions the mean squared Pearson coefficient is
larger for spikes than for bursts in every session; the mean ratio is 1.99 (SD 0.85) with 0.1 ms
bins and 2.13 (SD 1.65) with 0.4 ms bins when tau = 0 is left out, within 2.89 +/- 1.49, and 1.31
and 1.21 when it is included (outside). All 84 bursting sessions have bursts of n > 1, so reading
(b) of the sessions is the same.

## Information per burst (Figure 7): per-session measures

For every kept session, under the main burst reading, with the bin widths (a) 0.1 ms and (b) 0.4
ms: rbar_n, I_n(1) in bits per event (reading i; reading ii is T = 0.8 s times it), I'_n in bits/s,
and the jitter of each n (readings (a) and (b) of the events). Fits are over the n values present.

In [ ]:
T = ia1.T1 - ia1.T0
info = {}
for s in KEPT:
    for bw in [1e-4, 4e-4]:
        info[(s.name, bw)] = ia1.information(ia1.burst_rates(s.trials, LIM[s.name], bw))


def fit_rows(sel, bw):
    rows = []
    for s in sel:
        inf = info[(s.name, bw)]
        ns = np.array(sorted(inf))
        i1 = np.array([inf[n]["i1"] for n in ns])
        rbar = np.array([inf[n]["rbar"] for n in ns])
        rate = np.array([inf[n]["rate"] for n in ns])
        occ = rbar / rbar.sum()
        row = dict(session=s.name, protocol=s.protocol, n_max=ns.max(), n_values=len(ns))
        for name, y in [("i1", i1), ("i1_ii", i1 * T), ("occ_log2", np.log2(occ)), ("occ_ln", np.log(occ)),
                        ("occ_log10", np.log10(occ)), ("rate", rate)]:
            sl, se, pg, p2 = ia1.line_fit(ns, y)
            row.update({f"{name}_slope": sl, f"{name}_se": se, f"{name}_pg": pg, f"{name}_p2": p2})
        row["total_rate"] = rate.sum()
        row["frac_gt1"] = rate[ns > 1].sum() / rate.sum()
        rows.append(row)
    return pd.DataFrame(rows)


F7 = {bw: fit_rows(BURSTING, bw) for bw in [1e-4, 4e-4]}
print("bursting sessions:", len(BURSTING), "; with at least 3 n values (a slope with an error):",
      int((F7[1e-4].n_values >= 3).sum()), "; with 2:", int((F7[1e-4].n_values == 2).sum()), "; with 1:", int((F7[1e-4].n_values == 1).sum()))

## fig7ab-information-per-burst-increases

> The information carried by each n-burst increases with n; every bursting session has a
> significantly positive slope of I_n(1) against n, with an average slope of 1.5 bits (SD 0.7
> bits) over 59 bursting cells.

Significance (a) slope more than one standard error above zero, (b) one-sided t-test p < 0.05.
Sessions with fewer than three n values have no error for the slope and count as not significant.

In [ ]:
rows = []
for bw in [1e-4, 4e-4]:
    f = F7[bw]
    for norm, col in [("(i) bits per event", "i1"), ("(ii) Eq. 10 as printed", "i1_ii")]:
        for subset, g in [("(a) all bursting", f), ("(b) n_max > 1", f[f.n_max > 1])]:
            sig_a = (g[f"{col}_slope"] - g[f"{col}_se"] > 0)
            sig_b = (g[f"{col}_pg"] < 0.05)
            rows.append(dict(bin_ms=bw * 1e3, I1=norm, sessions=subset, n=len(g), mean_slope=round(g[f"{col}_slope"].mean(), 2),
                             sd_slope=round(g[f"{col}_slope"].std(ddof=1), 2), positive=int((g[f"{col}_slope"] > 0).sum()),
                             sig_a=int(sig_a.sum()), sig_b=int(sig_b.sum())))
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** The information per burst rises with n: the mean slope is 2.14
bits (SD 0.85) with 0.1 ms bins and 1.85 (SD 0.73) with 0.4 ms bins in bits per event, and 1.71 and
1.48 for Eq. 10 as printed, all within 1.5 +/- 0.7; all 84 slopes are positive, 83 are
significant by one standard error and 82 by the t-test (one session has only two n values, so no
error for its slope).
The population is 84 bursting sessions, outside 59 +/- 5.9.

## fig7cd-occurrence-decreases

> High-n bursts are rare; in every session the frequency of occurrence decreases significantly with
> n, with an average slope of -2.6 (SD 0.7).

Slope of the logarithm of rbar_n / sum rbar_n on n; logarithm (a) base 2, (b) natural, (c) base
10. Significance as in fig7ab (negative here). The occurrence does not depend on the bin width.

In [ ]:
f = F7[1e-4]
rows = []
for col in ["occ_log2", "occ_ln", "occ_log10"]:
    for subset, g in [("(a) all bursting", f), ("(b) n_max > 1", f[f.n_max > 1])]:
        rows.append(dict(log=col, sessions=subset, n=len(g), mean_slope=round(g[f"{col}_slope"].mean(), 2),
                         sd_slope=round(g[f"{col}_slope"].std(ddof=1), 2), negative=int((g[f"{col}_slope"] < 0).sum()),
                         sig_a=int((g[f"{col}_slope"] + g[f"{col}_se"] < 0).sum()), sig_b=int(((1 - g[f"{col}_pg"]) < 0.05).sum())))
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** The log2 occurrence falls with n with a mean slope of -2.71
(SD 1.09), within -2.6 +/- 0.7 (natural log -1.88, log10 -0.82, outside); every slope is negative
and 83 of 84 are significant by reading (a), 81 by reading (b), so not every session, as the claim
states, but above 90%.

## fig7ef-jitter-independent-of-n

> The jitter of the first spike of a burst is roughly independent of n in 80% of the bursting
> sessions, and the mean slope of jitter against n is -0.03 ms (SD 0.2 ms).

Jitter per n is the mean over events of the SD of the onset times. Events (a) peaks of the onset
histogram in 0.1 ms bins, between the troughs on either side; (b) onsets within the session's
first-peak domain (the limiting ISI) centred on an event centre, the centres being the histogram
bins taken from the highest down (`ia1.jitter_events_b`). Slopes over the n values with a jitter (events with at
least two onsets); "not significant" is (a) |slope| below one standard error, (b) two-sided p >=
0.05; sessions with fewer than three such n values have no test and are counted apart.

In [ ]:
rows = []
for s in BURSTING:
    tr, on, n = ia1.bursts(s.trials, LIM[s.name])
    row = dict(session=s.name)
    for ev in "ab":
        ns, js = [], []
        for m in np.unique(n):
            obt = [on[(n == m) & (tr == i)] for i in range(s.n_trials)]
            j = ia1.jitter_events_a(obt) if ev == "a" else ia1.jitter_events_b(obt, LIM[s.name])
            if np.isfinite(j):
                ns.append(m)
                js.append(j * 1e3)
        sl, se, pg, p2 = ia1.line_fit(ns, js)
        row.update({f"{ev}_slope": sl, f"{ev}_se": se, f"{ev}_p2": p2, f"{ev}_nvals": len(ns),
                    f"{ev}_j1": js[0] if ns and ns[0] == 1 else np.nan})
    rows.append(row)
J = pd.DataFrame(rows)
summ = []
for ev in "ab":
    g = J[J[f"{ev}_nvals"] >= 3]
    summ.append(dict(events=ev, sessions_with_test=len(g), no_test=int((J[f"{ev}_nvals"] < 3).sum()),
                     mean_slope_ms=round(J[f"{ev}_slope"].mean(), 3), sd_slope_ms=round(J[f"{ev}_slope"].std(ddof=1), 3),
                     pct_not_sig_a=round(100 * (g[f"{ev}_slope"].abs() < g[f"{ev}_se"]).mean(), 1),
                     pct_not_sig_b=round(100 * (g[f"{ev}_p2"] >= 0.05).mean(), 1),
                     median_jitter_n1_ms=round(J[f"{ev}_j1"].median(), 3)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: reproduced** (resting on how events are formed). With events read as onsets within the
first-peak domain around an event centre (b) the mean slope of jitter on n is 0.005 ms (SD 0.050),
within -0.03 +/- 0.2, and 87.7% of the 81 sessions with a test have a slope not significantly
different from 0 by the t-test, inside 80 +/- 8 (the bound is 88). With events between histogram
troughs (a) the slope is -0.001 ms but 91.6% are not significant (outside); with significance
read as one standard error, 43.4% and 34.6% (outside). The median jitter of 1-bursts is 0.389 ms
under (b) and 0.058 ms under (a).

## fig7gh-information-rate

> The information rate of n-bursts decreases with n (average slope -11 bits/s, SD 16 bits/s), yet
> bursts with n > 1 together carry on average 47% of the total information over the bursting cells.

I'_n = rbar_n I_n(1) (bits/s), the same under readings (i) and (ii) of I_n(1).

In [ ]:
rows = []
for bw in [1e-4, 4e-4]:
    f = F7[bw]
    for subset, g in [("(a) all bursting", f), ("(b) n_max > 1", f[f.n_max > 1])]:
        rows.append(dict(bin_ms=bw * 1e3, sessions=subset, n=len(g), mean_slope=round(g.rate_slope.mean(), 1),
                         sd_slope=round(g.rate_slope.std(ddof=1), 1), negative=int((g.rate_slope < 0).sum()),
                         mean_frac_gt1_pct=round(100 * g.frac_gt1.mean(), 1), sd_frac_pct=round(100 * g.frac_gt1.std(ddof=1), 1),
                         mean_total_bits_s=round(g.total_rate.mean(), 1)))
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** The information rate falls with n in all 84 sessions, but the mean
slope is -44.7 bits/s (SD 36.0) with 0.1 ms bins and -36.6 (SD 30.6) with 0.4 ms bins, outside -11
+/- 16, and bursts with n > 1 carry 36.3% and 38.6% of the information, outside 47 +/- 4.7. The
total rate averages 246.6 and 203.4 bits/s.

## direct-method-comparison

> Information rates from the burst code are highly correlated with those of the direct method (R =
> 0.95), averaging 191 against 222 bits/s, the direct method being higher in all sessions but one
> by 31 bits/s on average, so the burst code keeps about 85% of the information.

Direct method on binary words of 0.4 ms bins, words starting at every bin over 200 to 1000 ms,
entropies NSB-corrected (`ia1.nsb_entropy`). Readings (a) the rate at 8 bins (3.2 ms), (b) a
linear extrapolation in 1 / word length over 1 to 8 bins. The burst-code rate is I' of
fig7gh (non-bursting sessions as all 1-bursts), with bins (a) 0.1 ms and (b) 0.4 ms. Sessions (a)
all kept, (b) bursting.

In [ ]:
rows = []
for s in KEPT:
    rates = [ia1.direct_information(s.trials, L) for L in range(1, 9)]
    sl, ic, *_ = stats.linregress(1 / np.arange(1, 9), rates)
    row = dict(session=s.name, bursting=LIM[s.name] > 0, direct_8=rates[-1], direct_extrap=ic)
    for bw in [1e-4, 4e-4]:
        row[f"burst_{bw * 1e3:.1f}"] = sum(v["rate"] for v in info[(s.name, bw)].values())
    rows.append(row)
DM = pd.DataFrame(rows)
summ = []
for dcol in ["direct_8", "direct_extrap"]:
    for bcol in ["burst_0.1", "burst_0.4"]:
        for lab, g in [("(a) kept", DM), ("(b) bursting", DM[DM.bursting])]:
            diff = g[dcol] - g[bcol]
            summ.append(dict(direct=dcol, burst=bcol, sessions=lab, n=len(g), R=round(np.corrcoef(g[dcol], g[bcol])[0, 1], 3),
                             mean_direct=round(g[dcol].mean(), 1), sd_direct=round(g[dcol].std(ddof=1), 1),
                             mean_burst=round(g[bcol].mean(), 1), sd_burst=round(g[bcol].std(ddof=1), 1),
                             mean_diff=round(diff.mean(), 1), sd_diff=round(diff.std(ddof=1), 1),
                             direct_higher=f"{int((diff > 0).sum())}/{len(g)}",
                             lost_pct=round(100 * (1 - g[bcol].mean() / g[dcol].mean()), 1)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: partially reproduced.** With 0.4 ms bins for the burst code, over the 115 kept
sessions, R = 0.965, the direct method gives 204.3 bits/s (8 bins) and the burst code 183.6, a
mean difference of 20.7 bits/s, all within tolerance, and the burst code keeps 90% (10.1% lost,
outside 14 +/- 1.4). The direct method is higher in 94 of 115 sessions (82%), not in all but one.
Over the bursting sessions R = 0.950, 224.3 against 203.4 bits/s, difference 20.9, higher in 66 of
84. The extrapolated direct rate changes these by under 1 bit/s. With 0.1 ms bins the burst code
is above the direct method on average (227.3 against 204.3 bits/s).

## Burst-triggered averages: per-session measures

For each bursting session (main reading), the stimulus in dB (stimulus file shifted to the
intensity in `cell.dat`, as `stimtodB.m` does) around every burst onset in 200 to 1000 ms, from
-25 to +15 ms at 20 kHz (Methods, Estimation of burst-triggered averages). The central maximum of
an nBTA is its maximum over the window; tau_n is its distance to the onset (Eq. 13 and the
sentence after it: "The time tau_n between the maximum of nBTA and tau = 0").

In [ ]:
def p_two_sided_ttests(a, b):
    return stats.ttest_ind(a, b, axis=0).pvalue


BTA = {}
for s in BURSTING:
    st = stim_db(s)
    tr, on, n = ia1.bursts(s.trials, LIM[s.name])
    seg = {}
    for m in range(1, 5):
        if (n == m).sum() >= 2:
            seg[m], tt = ia1.segments(st, on[n == m])
    sp = spikes_in(s)
    sta_seg, tt = ia1.segments(st, sp)
    BTA[s.name] = dict(seg=seg, tt=tt, sta_seg=sta_seg, on=on, n=n, tr=tr, mean_stim=st[4000:20000].mean())
have4 = [s for s in BURSTING if len(BTA[s.name]["seg"]) == 4]
print("bursting sessions:", len(BURSTING), "; with at least two bursts of each n = 1 to 4:", len(have4))

## fig8-nbta-differ

> In every bursting cell the burst-triggered averages for n = 1 to 4 differ significantly (in the
> example cell their central peaks grow higher and wider with n); 50 of the 58 bursting sessions
> show significantly different central peaks in the time domain, the other eight having too few
> bursts.

Two-way ANOVAs (frequency bin x n) on the real and the imaginary parts of the Fourier transform of
each segment, each at 0.01 / 2; frequency bins (a) all, (b) up to the cutoff. Then at each time
point, t-tests of each nBTA against the three others, p < 0.01; a point is marked for n when all
three are significant. Central peak: from the minimum before to the minimum after the central
maximum. Readings (a) at least one marked point in the central peak of every n, (b) more than half
of the points of every central peak marked.

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    tt = B["tt"]
    segs = [B["seg"][m] for m in range(1, 5)]
    ffts = [np.fft.rfft(g, axis=1) for g in segs]
    freqs = np.fft.rfftfreq(len(tt), 1 / ia1.STIM_RATE)
    row = dict(session=s.name, protocol=s.protocol, counts=[len(g) for g in segs])
    for fr, sel in [("a", slice(None)), ("b", freqs <= s.cutoff)]:
        ps = []
        for part in [np.real, np.imag]:
            gs = [part(f)[:, sel] for f in ffts]
            if part is np.imag:
                gs = [g[:, 1:] for g in gs]          # the imaginary part of the 0 Hz bin is 0
            ps.append(min(ia1.two_way_anova(gs)))
        row[f"anova_{fr}"] = min(ps) < 0.01 / 2
    btas = [g.mean(0) for g in segs]
    marked = []
    for i in range(4):
        sig = np.ones(len(tt), bool)
        for j in range(4):
            if j != i:
                sig &= p_two_sided_ttests(segs[i], segs[j]) < 0.01
        ip, a, b = ia1.central_peak(btas[i], tt)
        marked.append(sig[a:b + 1].mean())
    row["peak_any"] = all(m > 0 for m in marked)
    row["peak_most"] = all(m > 0.5 for m in marked)
    row["marked_fraction"] = [round(m, 2) for m in marked]
    rows.append(row)
N8 = pd.DataFrame(rows)
print(N8.to_string(index=False))
for fr in "ab":
    g = N8[N8[f"anova_{fr}"]]
    print(f"\nfrequency bins ({fr}): ANOVA significant in {len(g)} of {len(N8)} sessions with n = 1 to 4;"
          f" central peaks differ (a) any point: {int(g.peak_any.sum())}, (b) most points: {int(g.peak_most.sum())}")
print("bursting sessions (detail 4 (a)):", len(BURSTING), "; with bursts of n = 1 to 4 (two or more each):", len(have4))

**Result: partially reproduced.** All 71 bursting sessions with at least two bursts of each n = 1
to 4 pass the frequency-domain ANOVA, with all frequency bins and with bins up to the cutoff. In
the time domain 60 of them have marked points in every central peak (a), outside 50 +/- 5, and 14
have most of every central peak marked (b). The 60 of 71 (85%) is the paper's proportion (50 of 58,
86%), but the bursting sessions number 84 (71 with n = 1 to 4), outside 58 +/- 5.8, because more
sessions are kept.

## fig8-latency-decreases

> The latency between the stimulus upstroke and burst onset decreases with n, so that high or wide
> deflections produce prompt responses with high-n bursts.

tau_n per session with bursts of n = 1 to 4. Readings (a) tau_n decreasing from n = 1 to 4 (tau_4 <
tau_1 with no increase between consecutive n) in the majority of those sessions; (b) the mean over
sessions of tau_n decreasing from 1 to 4.

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    taus = [-B["tt"][ia1.central_peak(B["seg"][m].mean(0), B["tt"])[0]] * 1e3 for m in range(1, 5)]
    rows.append(dict(session=s.name, protocol=s.protocol, **{f"tau{m}": t for m, t in zip(range(1, 5), taus)},
                     non_increasing=all(np.diff(taus) <= 0) and taus[3] < taus[0], tau4_below_tau1=taus[3] < taus[0]))
LT = pd.DataFrame(rows)
print(LT.round(2).to_string(index=False))
print("\n(a) sessions with tau_n decreasing from 1 to 4:", int(LT.non_increasing.sum()), "of", len(LT),
      "; with tau_4 < tau_1:", int(LT.tau4_below_tau1.sum()))
print("(b) mean tau_n (ms):", LT[["tau1", "tau2", "tau3", "tau4"]].mean().round(2).to_dict())
print("    Wilcoxon tau_1 vs tau_4: p =", stats.wilcoxon(LT.tau1, LT.tau4).pvalue)

**Result: reproduced.** The mean latency falls with n: 5.75, 4.13, 2.88 and -0.55 ms for n = 1 to
4 over 71 sessions (reading b); 43 of 71 sessions have tau_n decreasing from 1 to 4 (reading a),
all 71 have tau_4 < tau_1, and the Wilcoxon test of tau_1 against tau_4 gives p = 2.4e-13. A
negative tau_4 means the 4BTA reaches its maximum after the burst starts in some sessions.

## fig8e-sigma-minimum

> The standard deviation of the stimuli preceding an n-burst has a minimum at a position nearly the
> same for all n; over the bursting cells the SD of the minimum's position across n = 1 to 4
> averages 0.4 ms (SD 0.7 ms), roughly 0.05 times the inverse cutoff frequency.

sigma_n(tau) by Eq. 14. Window for the minimum (a) -25 to 0 ms, (b) -25 to +15 ms. The SD across
the four n is the sample SD (N - 1), as MATLAB's `std`; the population SD (N) is shown too.

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    tt = B["tt"]
    row = dict(session=s.name, cutoff=s.cutoff)
    for w, sel in [("a", tt <= 0), ("b", tt <= tt.max())]:
        pos = [tt[sel][np.argmin(B["seg"][m].std(0, ddof=1)[sel])] * 1e3 for m in range(1, 5)]
        row[f"pos_{w}"] = pos
        row[f"sd_{w}"] = np.std(pos, ddof=1)
        row[f"sd0_{w}"] = np.std(pos)
    rows.append(row)
SG = pd.DataFrame(rows)
for w in "ab":
    for col in [f"sd_{w}", f"sd0_{w}"]:
        x = SG[col]
        xc = SG[col] * SG.cutoff * 1e-3
        print(f"window ({w}) {'N-1' if col.startswith('sd_') else 'N'}: mean SD of position {x.mean():.3f} ms (SD {x.std(ddof=1):.3f}),"
              f" median {x.median():.3f}; times cutoff: mean {xc.mean():.4f} (SD {xc.std(ddof=1):.4f}), n = {len(x)}")
print("\nmedian position of the minimum (ms), window (a), per n:", np.median(np.array(SG.pos_a.tolist()), 0))

**Result: not reproduced.** The minimum of sigma_n lies near -6 ms for every n (medians -6.65,
-6.3, -5.0 and -6.25 ms), but the SD of its position across n = 1 to 4 averages 2.16 ms (SD 2.40)
in window (a) and 2.64 ms in window (b), outside 0.4 +/- 0.7, and 1.87 and 2.29 ms with the
population SD; times the cutoff it is 0.36 to 0.43, against 0.05.

## sigma-n-below-sta-sigma

> In 98% of bursting cells, for all n, the SD of the stimuli preceding an n-burst is smaller than
> the SD of the stimuli preceding all spikes, by a ratio of minima of 1.62 on average (SD 0.56).

"Smaller" read as (a) the minimum of sigma_n below the minimum of sigma, (b) sigma_n below sigma at
every tau in -25 to +15 ms; n = 1 to 4. The ratio is min sigma over the mean over n of min sigma_n,
minima over -25 to +15 ms (and over -25 to 0 ms as a variant).

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    tt = B["tt"]
    sig = B["sta_seg"].std(0, ddof=1)
    sn = [B["seg"][m].std(0, ddof=1) for m in range(1, 5)]
    rows.append(dict(session=s.name, a=all(x.min() < sig.min() for x in sn), b=all((x < sig).all() for x in sn),
                     ratio=sig.min() / np.mean([x.min() for x in sn]),
                     ratio_pre=sig[tt <= 0].min() / np.mean([x[tt <= 0].min() for x in sn])))
SB = pd.DataFrame(rows)
print(f"sessions: {len(SB)}; (a) min sigma_n < min sigma for all n: {100 * SB.a.mean():.1f}%; (b) at every tau: {100 * SB.b.mean():.1f}%")
print(f"ratio min sigma / mean_n min sigma_n: mean {SB.ratio.mean():.3f} (SD {SB.ratio.std(ddof=1):.3f}), range {SB.ratio.min():.3f} to {SB.ratio.max():.3f};"
      f" window -25 to 0 ms: mean {SB.ratio_pre.mean():.3f} (SD {SB.ratio_pre.std(ddof=1):.3f})")

**Result: reproduced.** In 98.6% of the 71 sessions the minimum of sigma_n is below the minimum of
sigma for all n = 1 to 4 (reading a), within 98 +/- 9.8, and the ratio of the minima averages 1.604
(SD 0.278), within 1.62 +/- 0.56. Read as sigma_n below sigma at every tau (b), it holds in 1.4%.

## fig9-nbta-not-concatenated-1bta

> The nBTA differs from n 1BTAs shifted by the burst's own ISIs and summed; for a large fraction of
> cells the difference is significant, the fraction diminishing as n grows; results are similar
> with shifted STAs.

f_n(t) = sum over the burst's spikes of 1BTA(t - t_i) (Eq. 15). Summing n copies of a stimulus in
dB only compares with the nBTA once the mean stimulus is subtracted (as the paper does for Figure
9C); this is taken as the reading of Eq. 15, an assumption, and the average of the shifted copies
(the sum over n) is shown as a variant. The 1BTA (or STA) is computed from -60 to +15 ms so that
its shifts cover the window. Comparison over the interval between the minima flanking the central
maximum of the nBTA: two-way ANOVAs (frequency bin x group) on the real and imaginary parts of the
Fourier transform, at 0.01 / 2, with observations (a) each burst's segment against its own f_n
(paired: ANOVA on the differences), (b) the segments and the f_n as two groups. Then t-tests at
each time point (p < 0.01) and sessions with more than 70% of the interval significant.

In [ ]:
def paired_anova(d):
    # Differences segment - f_n per burst (rows) and frequency bin (columns): p of a mean
    # difference (group main effect) and of its variation across bins (the interaction).
    Nb, F = d.shape
    m = d.mean()
    mf = d.mean(0)
    ss_err = ((d - mf) ** 2).sum()
    df_err = (Nb - 1) * F
    ms = ss_err / df_err
    p_main = stats.f.sf(Nb * F * m ** 2 / ms, 1, df_err)
    p_int = stats.f.sf(Nb * ((mf - m) ** 2).sum() / (F - 1) / ms, F - 1, df_err)
    return min(p_main, p_int)


rows = []
for s in BURSTING:
    B = BTA[s.name]
    st = stim_db(s)
    ms = B["mean_stim"]
    tr, on, n = B["tr"], B["on"], B["n"]
    long_t0 = 0.06
    one, tl = ia1.segments(st, on[n == 1], pre=long_t0)
    allsp, _ = ia1.segments(st, spikes_in(s), pre=long_t0)
    kernels = {"1BTA": one.mean(0) - ms, "STA": allsp.mean(0) - ms}
    i_zero = int(round(long_t0 * ia1.STIM_RATE))
    for m in [2, 3, 4]:
        if m not in B["seg"]:
            continue
        idx = np.flatnonzero(n == m)
        seg = B["seg"][m] - ms
        tt = B["tt"]
        ip, a, b = ia1.central_peak(seg.mean(0), tt)
        if b - a < 3:
            continue
        # spike offsets of each burst relative to its onset
        trial_spikes = s.trials
        for kname, ker in kernels.items():
            for scale in ["sum", "mean"]:
                fn = np.zeros_like(seg)
                for r, k in enumerate(idx):
                    t = trial_spikes[tr[k]]
                    j = np.searchsorted(t, on[k])
                    offs = t[j:j + m] - on[k]
                    for o in offs:
                        sh = int(round(o * ia1.STIM_RATE))
                        fn[r] += ker[i_zero - 500 - sh:i_zero + 300 - sh]
                    if scale == "mean":
                        fn[r] /= m
                x, y = seg[:, a:b + 1], fn[:, a:b + 1]
                X, Y = np.fft.rfft(x, axis=1), np.fft.rfft(y, axis=1)
                pa = min(paired_anova(np.real(X - Y)), paired_anova(np.imag(X - Y)[:, 1:]) if X.shape[1] > 2 else 1)
                pb = min(min(ia1.two_way_anova([np.real(X), np.real(Y)])), min(ia1.two_way_anova([np.imag(X)[:, 1:], np.imag(Y)[:, 1:]])) if X.shape[1] > 2 else 1)
                tp = stats.ttest_ind(x, y, axis=0).pvalue
                rows.append(dict(session=s.name, n=m, kernel=kname, scale=scale, bursts=len(idx), anova_a=pa < 0.005, anova_b=pb < 0.005,
                                 t_frac=(tp < 0.01).mean()))
N9 = pd.DataFrame(rows)
summ = []
for (kname, scale, m), g in N9.groupby(["kernel", "scale", "n"]):
    summ.append(dict(kernel=kname, scale=scale, n=m, sessions=len(g), anova_a_pct=round(100 * g.anova_a.mean(), 1),
                     anova_b_pct=round(100 * g.anova_b.mean(), 1),
                     t70_a_pct=round(100 * (g.anova_a & (g.t_frac > 0.7)).mean(), 1),
                     t70_b_pct=round(100 * (g.anova_b & (g.t_frac > 0.7)).mean(), 1)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: reproduced** (resting on subtracting the mean stimulus before summing the 1BTAs). The
nBTA differs from the summed 1BTAs by the ANOVA in 100% of the sessions for n = 2, 3 and 4, with
paired (a) and grouped (b) observations, so the fraction does not grow with n; the share with more
than 70% of the interval significant falls from 94.0% (n = 2) to 86.7% and 54.9% (n = 4). With the
STA the ANOVA fractions are 100% (97.2% for n = 4 grouped) and the 70% share falls from 100% to
40.8%; with the average of the shifted copies in place of their sum the results are alike.

## fig9c-population-nbta

> In the seven cells driven at 100 Hz cutoff and 6 dB, the population-average nBTAs show that
> bursts with more spikes follow higher or wider stimulus deflections, with large spread across
> cells.

Each session's 1BTA, 2BTA and 3BTA minus its mean stimulus, shifted so that its central maximum
sits at the mean tau_n over the sessions (shifting by tau_n, averaging and shifting back), averaged
over the bursting protocol-5 sessions with at least two bursts of n = 1 to 3. Width (a) full width
at half height, (b) between the minima flanking the central maximum.

In [ ]:
p5 = [s for s in BURSTING if s.protocol == 5 and all(m in BTA[s.name]["seg"] for m in (1, 2, 3))]
print("protocol-5 sessions: kept", sum(s.protocol == 5 for s in KEPT), "; bursting", sum(s.protocol == 5 for s in BURSTING),
      "; with n = 1 to 3:", len(p5), [s.name for s in p5])
fig, ax = plt.subplots(figsize=(6, 4))
rows = []
for m in (1, 2, 3):
    tt = BTA[p5[0].name]["tt"]
    btas, peaks = [], []
    for s in p5:
        b = BTA[s.name]["seg"][m].mean(0) - BTA[s.name]["mean_stim"]
        btas.append(b)
        peaks.append(ia1.central_peak(b, tt)[0])
    target = int(round(np.mean(peaks)))
    aligned = np.array([np.roll(b, target - p) for b, p in zip(btas, peaks)])
    avg = aligned.mean(0)
    ip, a, b2 = ia1.central_peak(avg, tt)
    rows.append(dict(n=m, height_dB=avg[ip], fwhm_ms=ia1.fwhm(avg, ip, 0.0) / 20, minima_width_ms=(b2 - a) / 20,
                     spread_at_peak_dB=aligned[:, ip].std(ddof=1), tau_ms=-tt[ip] * 1e3))
    ax.plot(tt * 1e3, avg, label=f"{m}BTA")
    ax.fill_between(tt * 1e3, avg - aligned.std(0, ddof=1), avg + aligned.std(0, ddof=1), alpha=0.2)
ax.set(xlabel="time from burst onset (ms)", ylabel="stimulus minus its mean (dB)")
ax.legend()
plt.show()
F9C = pd.DataFrame(rows)
print(F9C.round(2).to_string(index=False))

**Result: reproduced.** Over the 10 bursting protocol-5 sessions (the paper has seven) the
population-average central peak is 3.22, 7.11 and 8.60 dB above the mean stimulus for n = 1, 2
and 3, and its width grows from 5.10 to 5.50 to 7.15 ms at half height (12.70, 13.50, 17.15 ms
between minima). The SD across sessions at the peak is 0.57, 0.74 and 1.28 dB.

## fig10-burst-n-vs-height

> The probability of a burst with n spikes after a stimulus maximum depends on its height, low
> excursions giving no response or an isolated spike and high ones doublets, triplets or 4-bursts;
> all bursting cells show this.

Methods (Relating burst probabilities to the height of stimulus excursions), per trial: each
n-burst shifted back by tau_n; each local maximum of the stimulus in 200 to 1000 ms matched to the
burst whose shifted onset is closest within [t0 - T, t0 + T]; a burst matched to more than one
maximum kept by the nearest, the other maxima counting as no response. T is the width of the
central STA peak, (a) between its flanking minima, (b) full width at half height. P(n | h) in
windows of half-width 5% of the span of h, at 41 centres. Segregation (a) the h at which P(n | h)
peaks increases with n for n = 0 to the largest n with bursts (up to 4); (b) the mean of h
weighted by P(n | h) increases with n. Bursting sessions with bursts of n = 1 and 2 at least.

In [ ]:
from scipy.signal import argrelmax


def p_n_given_h(s, Twin):
    B = BTA[s.name]
    st = stim_db(s)
    i0, i1 = int(ia1.T0 * ia1.STIM_RATE), int(ia1.T1 * ia1.STIM_RATE)
    mx = argrelmax(st[i0:i1])[0] + i0
    tmax, h = mx / ia1.STIM_RATE, st[mx]
    tt = B["tt"]
    tau = {m: -tt[ia1.central_peak(B["seg"][m].mean(0), tt)[0]] for m in B["seg"]}
    resp = []
    for i in range(s.n_trials):
        k = B["tr"] == i
        on, n = B["on"][k], B["n"][k]
        sh = on - np.array([tau.get(m, tau[max(tau)]) for m in n])
        best = np.full(len(tmax), -1)
        for j, t0 in enumerate(tmax):
            d = np.abs(sh - t0)
            if len(d) and d.min() <= Twin:
                best[j] = np.argmin(d)
        r = np.zeros(len(tmax), int)
        for bi in np.unique(best[best >= 0]):
            js = np.flatnonzero(best == bi)
            jbest = js[np.argmin(np.abs(tmax[js] - sh[bi]))]
            r[jbest] = n[bi]
        resp.append(r)
    resp = np.concatenate(resp)
    matched = int((resp > 0).sum())
    hh = np.tile(h, s.n_trials)
    span = hh.max() - hh.min()
    centres = np.linspace(hh.min(), hh.max(), 41)
    nmax = min(4, resp.max())
    P = np.array([[np.mean(resp[np.abs(hh - c) <= 0.05 * span] == m) if (np.abs(hh - c) <= 0.05 * span).any() else np.nan
                   for c in centres] for m in range(nmax + 1)])
    return centres, P, nmax, matched


rows = []
example = None
for s in BURSTING:
    B = BTA[s.name]
    if not (1 in B["seg"] and 2 in B["seg"]):
        continue
    tt = B["tt"]
    sta = B["sta_seg"].mean(0) - B["mean_stim"]
    ip, a, b = ia1.central_peak(sta, tt)
    for tw, Tw in [("a", (b - a) / ia1.STIM_RATE), ("b", ia1.fwhm(sta, ip, 0.0) / ia1.STIM_RATE)]:
        centres, Pm, nmax, matched = p_n_given_h(s, Tw)
        peak_h = [centres[np.nanargmax(p)] for p in Pm]
        mean_h = [np.nansum(centres * p) / np.nansum(p) for p in Pm]
        rows.append(dict(session=s.name, T=tw, T_ms=Tw * 1e3, nmax=nmax, matched=matched, peak_h=np.round(peak_h, 1),
                         mean_h=np.round(mean_h, 1), a=all(np.diff(peak_h) > 0), b=all(np.diff(mean_h) > 0),
                         a3=all(np.diff(peak_h[:4]) > 0), b3=all(np.diff(mean_h[:4]) > 0)))
        if example is None and nmax == 4 and tw == "a":
            example = (s.name, centres, Pm)
H = pd.DataFrame(rows)
for tw in "ab":
    g = H[H["T"] == tw]
    print(f"T ({tw}): median T {g.T_ms.median():.2f} ms; sessions {len(g)}; segregation (a) peak h increasing: {int(g.a.sum())}"
          f" ({100 * g.a.mean():.1f}%); (b) mean h increasing: {int(g.b.sum())} ({100 * g.b.mean():.1f}%);"
          f" variant n = 0 to 3 only: (a) {int(g.a3.sum())}, (b) {int(g.b3.sum())}; maxima matched to a burst, median {g.matched.median():.0f}")
ga, gb = H[H["T"] == "a"].set_index("session"), H[H["T"] == "b"].set_index("session")
print("sessions where the two T readings match a different number of maxima:", int((ga.matched != gb.matched).sum()), "of", len(ga))
print(H[H["T"] == "a"].head(10).to_string(index=False))
name, centres, Pm = example
fig, ax = plt.subplots(figsize=(6, 4))
for m, p in enumerate(Pm):
    ax.plot(centres, p, label=f"n = {m}")
ax.set(xlabel="height of the stimulus maximum h (dB)", ylabel="P(n | h)", title=name, ylim=(0, 1))
ax.legend()
plt.show()

**Result: not reproduced.** The h at which P(n | h) peaks increases from n = 0 to the largest n
(up to 4) in 8 of 84 bursting sessions (9.5%) under both widths T, and the P-weighted mean h
increases in 21 (25.0%), against all bursting cells. Restricted to n = 0 to 3, a variant outside
the criterion, the ordering holds in 44 and 48 of 84 sessions: the 4-bursts, which are few, most
often break it.

## firing-rate-no-effect-on-limit

> The firing rate has practically no effect on the limiting ISI; a 50 Hz increase in rate shifts it
> by less than 0.4 ms and raises the average n by less than 25%.

Reading (a), the same cell and stimulus at two target rates 50 Hz apart: the headers record target
rates of 100 and 90 Hz only (Data check), so no pair exists. Reading (b): linear regressions over
the kept sessions of the limiting ISI (bursting sessions) and of the average n on the session's
rate, scaled to 50 Hz; the increase of the average n is relative to its mean.

In [ ]:
rows = []
for s in KEPT:
    c = ia1.burst_counts(s.trials, LIM[s.name])
    rows.append(dict(session=s.name, protocol=s.protocol, rate=D.loc[D.session == s.name, "rate"].iloc[0],
                     limit_ms=LIM[s.name] * 1e3, avg_n=ia1.average_n(c)[0]))
RT = pd.DataFrame(rows)
pairs = D[D.protocol.notna()].groupby(["cell", "stimulus"]).desired_rate.agg(lambda x: sorted(set(x.dropna())))
print("cell and stimulus with more than one target rate:", [k for k, v in pairs.items() if len(v) > 1])
for lab, g in [("all kept", RT), ("protocol 1 only", RT[RT.protocol == 1])]:
    gb = g[g.limit_ms > 0]
    r1 = stats.linregress(gb.rate, gb.limit_ms)
    r2 = stats.linregress(g.rate, g.avg_n)
    print(f"{lab}: limiting ISI slope {r1.slope:.4f} ms/Hz (p {r1.pvalue:.3g}, n {len(gb)}) -> shift for +50 Hz {50 * r1.slope:.3f} ms;"
          f" average n slope {r2.slope:.5f} per Hz (p {r2.pvalue:.3g}, n {len(g)}) -> +50 Hz changes it by {100 * 50 * r2.slope / g.avg_n.mean():.1f}%"
          f" of its mean {g.avg_n.mean():.3f}")

**Result: reproduced** (reading b). No cell was recorded at two target rates (the headers hold 100
and 90 Hz only), so reading (a) cannot be formed. Across the kept sessions a 50 Hz higher rate goes
with a limiting ISI shorter by 0.189 ms (p = 0.488, 84 bursting sessions), below 0.44 ms, and an
average n higher by 13.6% (p = 0.0688), below 27.5%; within protocol 1 alone, -0.007 ms and
20.3%. These are differences between cells, not the within-cell change the paper may have
measured.

## model-reproduces-coding-trends (blocked)

> The threshold-linear model with refractoriness reproduces the coding trends of the data, at
> single-cell and population level, though it is less precise in time than the real cells.

Depends on model-matches-classification, which is blocked (`data_missing`, no thresholds s_TH).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-sessions-and-trials | partially reproduced |
| table1-sessions-per-protocol | not reproduced |
| firing-rate-uniform | reproduced |
| threshold-population | blocked: data missing |
| table1-bursting-sessions | not reproduced |
| table1-isolated-spikes | partially reproduced (assumed) |
| table1-burst-length | not reproduced |
| fig5-burst-probability-vs-stimulus | reproduced (assumed) |
| fig3-correlation-shape | reproduced (assumed) |
| method-correlation-error | reproduced (assumed) |
| model-matches-classification | blocked: data missing |
| fig6-bursts-less-correlated | reproduced (assumed) |
| fig7ab-information-per-burst-increases | partially reproduced (assumed) |
| fig7cd-occurrence-decreases | partially reproduced (assumed) |
| fig7ef-jitter-independent-of-n | reproduced (assumed) |
| fig7gh-information-rate | not reproduced (assumed) |
| direct-method-comparison | partially reproduced (assumed) |
| fig8-nbta-differ | partially reproduced (assumed) |
| fig8-latency-decreases | reproduced (assumed) |
| fig8e-sigma-minimum | not reproduced (assumed) |
| sigma-n-below-sta-sigma | reproduced (assumed) |
| fig9-nbta-not-concatenated-1bta | reproduced (assumed) |
| fig9c-population-nbta | reproduced (assumed) |
| fig10-burst-n-vs-height | not reproduced (assumed) |
| firing-rate-no-effect-on-limit | reproduced (assumed) |
| model-reproduces-coding-trends | blocked: data missing |

How the analysis differs from the paper's:

- Population. The paper keeps 86 of its 132 sessions by the rate SD (< 35 Hz) and a rate near
  100 Hz. In the data set that rule keeps 115 of the 116 sessions of the paper's seven protocols,
  so every population result here is over more sessions than the paper's (60 reference sessions
  against 43). The paper's other selection is not stated.
- Burst identification. The Methods leave open which maximum is the first peak, which minimum
  bounds it and what sigma and N are in the error bar. The first local maximum (peak reading a)
  falls on sampling fluctuations of the rising flank and leaves 15 of 115 sessions bursting; with
  N the number of trials the error bars are so wide that no session bursts. The main reading is
  the largest value below 5 ms, the first significant minimum after it, and N the number of product
  terms, under which the error bar is 2.7% of the span of C_s (the paper's 3.4%). Outcomes that
  rest on bursts are marked as resting on this choice.
- Thresholds. The data set holds no threshold s_TH, so the model of Eq. 2 and the claims on it
  are blocked.
- Smaller choices are stated in each section: the stimulus is the AM signal in dB from
  `stimtodB.m` with the intensity of `cell.dat`; f_n of Eq. 15 sums 1BTAs after subtracting the
  mean stimulus; peaks of C_s are counted with a prominence threshold.

What was found about the dataset:

- Data1 holds 150 sessions (75 experiments of two stimuli each) of 45 cells; 116 sessions from
  37 cells on 23 dates use the paper's seven protocols, the paper's 37 cells; 28 sessions use the
  `gaussModality` and `lmd` stimuli and 6 use cutoffs of 50 and 1600 Hz, which the paper does not
  mention. Nothing marks the 86 sessions the paper kept.
- Data1 records no animal, no threshold and no best frequency; `cell.dat` gives the intensity
  played. 28 sessions were sorted off-line and have no header.
- The intensity in three headers differs from `cell.dat` by more than 1 dB (up to 6.6 dB); the
  descriptor says 98 to 533 trials, the files hold 69 to 503; six sessions had pauses of 450 ms,
  not 700. Two experiments (03-01-15-aa, 03-06-27-ae) are missing from `singlecells.txt`.
- Spikes and stimuli share the trial clock: the STA peaks 5.6 ms (median) before the spike in
  every session and almost vanishes with the stimulus shifted by 100 ms.